**Install** the required libraries for object detection, tracking, and video processing

In [ ]:
!pip install -q supervision "ultralytics<=8.3.40"

In [ ]:
import cv2
import numpy as np
import supervision as sv
from tqdm import tqdm
from ultralytics import YOLO
from collections import defaultdict, deque
import os

In [ ]:
# Replace with the path to your own input video
SOURCE_VIDEO_PATH = "traffic_video.mp4"
TARGET_VIDEO_PATH = "vehicles-result.mp4"
CONFIDENCE_THRESHOLD = 0.3
IOU_THRESHOLD = 0.5
MODEL_NAME = "yolov8x.pt"
MODEL_RESOLUTION = 1280

Define the ROI coordinates and real-world road dimensions **(Update these values according to your own video)**

In [ ]:
SOURCE = np.array([
    [438, 224],     # A
    [774, 223],     # B
    [1650, 719],    # C
    [-100, 719]     # D
])

ROAD_WIDTH = 17.65
ROAD_LENGTH = 60

TARGET = np.array([
    [0, 0],
    [ROAD_WIDTH - 1, 0],
    [ROAD_WIDTH - 1, ROAD_LENGTH - 1],
    [0, ROAD_LENGTH - 1],
])

In [ ]:
frame_generator = sv.get_video_frames_generator(source_path=SOURCE_VIDEO_PATH)
frame_iterator = iter(frame_generator)
frame = next(frame_iterator)

In [ ]:
annotated_frame = frame.copy()
annotated_frame = sv.draw_polygon(scene=annotated_frame, polygon=SOURCE, color=sv.Color.RED, thickness=4)
sv.plot_image(annotated_frame)

Create the View Transformer for mapping points from the camera view to the bird's-eye view

In [ ]:
class ViewTransformer:

    def __init__(self, source: np.ndarray, target: np.ndarray) -> None:
        source = source.astype(np.float32)
        target = target.astype(np.float32)
        self.m = cv2.getPerspectiveTransform(source, target)

    def transform_points(self, points: np.ndarray) -> np.ndarray:
        if points.size == 0:
            return points

        reshaped_points = points.reshape(-1, 1, 2).astype(np.float32)
        transformed_points = cv2.perspectiveTransform(reshaped_points, self.m)
        return transformed_points.reshape(-1, 2)

In [ ]:
view_transformer = ViewTransformer(source=SOURCE, target=TARGET)

In [ ]:
model = YOLO(MODEL_NAME)

video_info = sv.VideoInfo.from_video_path(video_path=SOURCE_VIDEO_PATH)
frame_generator = sv.get_video_frames_generator(source_path=SOURCE_VIDEO_PATH)

# tracer initiation
byte_track = sv.ByteTrack(
    frame_rate=video_info.fps,
    track_activation_threshold=CONFIDENCE_THRESHOLD
)

# annotators configuration
thickness = sv.calculate_optimal_line_thickness(
    resolution_wh=video_info.resolution_wh
)
text_scale = sv.calculate_optimal_text_scale(
    resolution_wh=video_info.resolution_wh
)
bounding_box_annotator = sv.BoxAnnotator(
    thickness=thickness
)
label_annotator = sv.LabelAnnotator(
    text_scale=text_scale,
    text_thickness=thickness,
    text_position=sv.Position.BOTTOM_CENTER
)

polygon_zone = sv.PolygonZone(
    polygon=SOURCE
)

coordinates = defaultdict(lambda: deque(maxlen=int(video_info.fps)))

# open target video
with sv.VideoSink(TARGET_VIDEO_PATH, video_info) as sink:

    # loop over source video frame
    for frame in tqdm(frame_generator, total=video_info.total_frames):

        result = model(frame, imgsz=MODEL_RESOLUTION, verbose=False)[0]
        detections = sv.Detections.from_ultralytics(result)

        # filter out detections by class and confidence
        detections = detections[detections.confidence > CONFIDENCE_THRESHOLD]
        detections = detections[detections.class_id != 0]

        # filter out detections outside the zone
        detections = detections[polygon_zone.trigger(detections)]

        # refine detections using non-max suppression
        detections = detections.with_nms(IOU_THRESHOLD)

        # pass detection through the tracker
        detections = byte_track.update_with_detections(detections=detections)

        # pixel-space anchor points (used for drawing connecting lines on frame)
        points_px = detections.get_anchors_coordinates(
            anchor=sv.Position.BOTTOM_CENTER
        )

        # real-world (transformed) points (used for speed + distance calculations)
        points = view_transformer.transform_points(points=points_px).astype(int)

        # store detections position
        for tracker_id, [_, y] in zip(detections.tracker_id, points):
            coordinates[tracker_id].append(y)

        # calculate nearest-neighbor distance (in meters) between detected objects
        nearest_distances = np.array([])
        nearest_pairs = []

        if len(points) > 1:
            dist_matrix = np.linalg.norm(
                points[:, None, :] - points[None, :, :], axis=-1
            )
            np.fill_diagonal(dist_matrix, np.inf)
            nearest_idx = dist_matrix.argmin(axis=1)
            nearest_distances = dist_matrix.min(axis=1)

            # build unique pairs so each connection is drawn only once
            seen_pairs = set()
            for i, j in enumerate(nearest_idx):
                pair = frozenset((i, j))
                if pair not in seen_pairs:
                    seen_pairs.add(pair)
                    nearest_pairs.append((i, j, nearest_distances[i]))

        # format labels (speed only, no tracker id)
        labels = []

        for i, tracker_id in enumerate(detections.tracker_id):
            if len(coordinates[tracker_id]) < video_info.fps / 2:
                labels.append("")
            else:
                # calculate speed
                coordinate_start = coordinates[tracker_id][-1]
                coordinate_end = coordinates[tracker_id][0]
                distance = abs(coordinate_start - coordinate_end)
                time = len(coordinates[tracker_id]) / video_info.fps
                speed = distance / time * 3.6
                labels.append(f"{int(speed)} km/h")

        # annotate frame
        annotated_frame = frame.copy()

        # draw connecting lines + distance labels between nearest object pairs
        for i, j, dist in nearest_pairs:
            pt1 = tuple(points_px[i].astype(int))
            pt2 = tuple(points_px[j].astype(int))

            cv2.line(annotated_frame, pt1, pt2, (0, 255, 255), max(1, thickness // 2))

            midpoint = ((pt1[0] + pt2[0]) // 2, (pt1[1] + pt2[1]) // 2)
            text = f"{dist:.1f}m"
            (text_w, text_h), _ = cv2.getTextSize(
                text, cv2.FONT_HERSHEY_SIMPLEX, text_scale, thickness
            )
            cv2.rectangle(
                annotated_frame,
                (midpoint[0] - 4, midpoint[1] - text_h - 4),
                (midpoint[0] + text_w + 4, midpoint[1] + 4),
                (0, 200, 200),
                -1
            )
            cv2.putText(
                annotated_frame, text, (midpoint[0], midpoint[1]),
                cv2.FONT_HERSHEY_SIMPLEX, text_scale, (0, 0, 0), thickness
            )

        annotated_frame = bounding_box_annotator.annotate(
            scene=annotated_frame, detections=detections
        )
        annotated_frame = label_annotator.annotate(
            scene=annotated_frame, detections=detections, labels=labels
        )

        # add frame to target video
        sink.write_frame(annotated_frame)



# Convert the output video to H.264 format for better compatibility

output_video_path = TARGET_VIDEO_PATH.replace(".mp4", "_output.mp4")
os.system(f'ffmpeg -y -i "{TARGET_VIDEO_PATH}" -vcodec libx264 -acodec aac "{output_video_path}"')

print(f"Output video saved at: {output_video_path}")

# Automatically download the output video file in Google Colab

try:
    from google.colab import files
    files.download(output_video_path)
except ImportError:
    pass